# Does giving the action a path into the STATE fix closed-loop rollout? (ablation)

**Direction:** `endogenous-action-interactive-world` · GRU · 2026-07-29. Registry: `ENDOGENOUS_RUNS.md`.

## The bug this tests

The endogenous actor fed its own action **only to the decoder**, never to the recurrence:

```
h_t     = GRU( enc(o_t) , h_{t-1} )              <- observation only; action absent
o^_{t+1} = dec( [ h_t , proj_dec(a_t) ] )         <- the action enters HERE, and only here
```

Consequence, measured in cell [2]: feeding **opposite actions produces a bit-identical next state**. The action
could not influence the imagined state at all — only the decoded observation — so its effect had to re-enter the
state through `decoder -> predicted obs -> encoder`, a lossy bottleneck. Every standard action-conditioned world
model (including this repo's own `action_gru_continuous`) puts the action in the transition. This was an
implementation error, not a property of endogenous action.

## The fix

```
h_t     = GRU( [ enc(o_t) , proj_trans(a_{t-1}) ] , h_{t-1} )   <- action now shapes the STATE
o^_{t+1} = dec( [ h_t , proj_dec(a_t) ] )                        <- unchanged
```

The **previous** action, because `a_t = pi(h_t)` is produced *from* `h_t` (using `a_t` would be circular);
`a_{t-1}` is what caused the transition into `t`. A **separate** projection from the decoder's, so decoder
behaviour is untouched. Enabled by `EndogenousActorConfig.action_in_transition` (default `False`, so existing
checkpoints load unchanged).

## Runs compared (single variable)

| code | descriptive label | action in transition | everything else |
|---|---|---|---|
| `L3s0` | L3 force+goal · strong 512h · seed 0 | **no** | identical |
| `L3s0_ait` | L3 force+goal · strong 512h · seed 0 · **action-in-transition** | **yes** | identical |

Both: level 3 (`force` dynamics, lethal, survival goal via REINFORCE), 512 hidden, 2-layer MLP encoder +
residual MLP decoder, 5-step free-run loss, 25000 iterations, seed 0, `obs_noise_std=0.05`.

> **⚠ Both runs still contain the hidden-state reset** (the GRU state is zeroed every 48 frames while the world
> continues), so a *null* here is partially confounded: the reset could mask the benefit. A *positive* result would
> have been clean. Read the verdict with that asymmetry in mind.

In [ ]:
# [1] Setup.
import sys, argparse
sys.path.insert(0, '../../../..')
import numpy as np, torch
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, Markdown, Image
from scripts.play import (Emulator, ModelDriver, AutoregressiveModelDriver, build_world)
from scripts.eval_editability_endogenous import load as load_ckpt
from pim.world_models.actor_gru import EndogenousActorGRU, EndogenousActorConfig
OK = {'blue':'#0072B2','orange':'#E69F00','green':'#009E73','grey':'#8a8f98','red':'#D55E00'}
def style_ax(ax):
    for s in ['top','right']: ax.spines[s].set_visible(False)
    ax.grid(alpha=0.25, lw=0.6)
ROOT = Path('../../../../runs/endogenous')
ANIM = ROOT / 'animations'
RUNS = [('L3s0', 'action NOT in transition'), ('L3s0_ait', 'action IN transition')]
def world_for(ck, seed=3):
    ic, sim = ck['interactive_cfg'], ck['sim_cfg']
    return build_world(argparse.Namespace(
        dynamics=ic['dynamics'], n_objects=2, seed=seed, obs_res=sim['obs_res'],
        obs_noise=sim['obs_noise_std'], death_on_collision=ic['death_on_collision'],
        death_on_wall=ic['death_on_wall'], reset_noise_frames=ic['reset_noise_frames']))
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# [2] The bug, demonstrated: can the action change the next hidden state at all?
rows = ['| architecture | ‖h(action=+1) − h(action=−1)‖ | interpretation |', '|---|---|---|']
for ait in [False, True]:
    m = EndogenousActorGRU(EndogenousActorConfig(
        input_dim=32, hidden_size=48, action_in_transition=ait)).eval()
    o = torch.rand(1, 32)
    with torch.no_grad():
        hA, _ = m.gru_step(o, None, prev_action=torch.full((1,2,2),  1.0))
        hB, _ = m.gru_step(o, None, prev_action=torch.full((1,2,2), -1.0))
    d = float((hA - hB).norm())
    rows.append(f"| action_in_transition={ait} | **{d:.4f}** | "
                f"{'the action shapes the state' if d > 0 else 'the action CANNOT affect the state'} |")
display(Markdown('\n'.join(rows)))

In [ ]:
# [3] Fig 1 — does the fix repair closed-loop control? Death rate under two control modes.
#     teacher-forced = the model sees the real observation each step (one-step prediction).
#     closed loop    = after a 15-frame warm-up it consumes only its OWN predictions.
res = {}
for tag, lab in RUNS:
    actor, observer, sim, ck = load_ckpt(ROOT / tag / 'ckpt_final.pt')
    res[tag] = {'label': lab, 'ait': ck['model_cfg'].get('action_in_transition', False)}
    for mode, mk in [('teacher-forced', lambda m: ModelDriver(m)),
                     ('closed-loop', lambda m: AutoregressiveModelDriver(m, warmup=15))]:
        deaths = 0
        for seed in range(6):
            w = world_for(ck, seed=seed); drv = mk(actor)
            for _ in range(300): w.step(drv.act(w))
            deaths += w.deaths
        res[tag][mode] = 1000 * deaths / (6 * 300)
NO_GOAL = 79.1   # strong no-goal control (L2s0), from eval_metrics.json
fig, ax = plt.subplots(figsize=(7.5, 4.0))
x = np.arange(2); w_ = 0.36
for j, (tag, lab) in enumerate(RUNS):
    ax.bar(x + (j-0.5)*w_, [res[tag]['teacher-forced'], res[tag]['closed-loop']], w_,
           label=f"{lab}", color=[OK['orange'], OK['blue']][j])
ax.axhline(NO_GOAL, color=OK['red'], ls='--', lw=1.3,
           label=f'no-goal control ({NO_GOAL:.0f}) — i.e. no policy at all')
ax.set_xticks(x); ax.set_xticklabels(['teacher-forced\n(sees the real world)', 'CLOSED LOOP\n(its own predictions)'])
ax.set_ylabel('deaths per 1000 frames  (↓ better)'); style_ax(ax); ax.legend(fontsize=8)
ax.set_title('Fig 1 — the fix helps closed-loop control only marginally', fontsize=11)
fig.tight_layout(); display(fig); plt.close(fig)
display(Markdown('| run | teacher-forced | closed-loop |\n|---|---|---|\n' + '\n'.join(
    f"| {res[t]['label']} | {res[t]['teacher-forced']:.1f} | **{res[t]['closed-loop']:.1f}** |" for t,_ in RUNS)))

In [ ]:
# [4] Fig 2 — the decisive measure: how fast does the model's IMAGINATION diverge from reality?
#     Reference lines: copy-previous-frame and random-independent-frame baselines
#     (pim/eval/baselines.py convention). Reaching the random line = the dream is uninformative.
STEPS = 60
curves = {}
for tag, lab in RUNS:
    actor, observer, sim, ck = load_ckpt(ROOT / tag / 'ckpt_final.pt')
    errs = np.zeros(STEPS); n = 0
    for seed in range(8):
        w = world_for(ck, seed=seed); drv = AutoregressiveModelDriver(actor, warmup=15)
        for _ in range(15): w.step(drv.act(w))          # warm-up on real observations
        for k in range(STEPS):
            act = drv.act(w); imagined = drv.last_pred.copy()
            real, _ = w.step(act)
            errs[k] += float(np.sqrt(((imagined - real) ** 2).mean()))
        n += 1
    curves[tag] = errs / n
# dataset baselines on a matched trace
_, _, sim0, ck0 = load_ckpt(ROOT / 'L3s0' / 'ckpt_final.pt')
w = world_for(ck0, seed=99); tr = [w.step(np.zeros((2,2)))[0].copy() for _ in range(400)]
tr = np.stack(tr)
identity = float(np.sqrt(((tr[1:] - tr[:-1]) ** 2).mean()))
random_b = float(np.sqrt(2.0 * tr.var()))
fig, ax = plt.subplots(figsize=(8.5, 4.2))
for (tag, lab), c in zip(RUNS, [OK['orange'], OK['blue']]):
    ax.plot(range(1, STEPS+1), curves[tag], lw=1.8, color=c, label=lab)
ax.axhline(identity, color=OK['grey'], ls='--', lw=1.2, label=f'copy-previous-frame baseline ({identity:.3f})')
ax.axhline(random_b, color=OK['red'], ls=':', lw=1.4, label=f'random independent frame ({random_b:.3f})')
ax.set_xlabel('closed-loop rollout step'); ax.set_ylabel('imagined-vs-real observation RMSE')
ax.set_title('Fig 2 — the imagination decouples from reality within ~10–20 steps', fontsize=11)
style_ax(ax); ax.legend(fontsize=8)
fig.tight_layout(); display(fig); plt.close(fig)

## Animations

Generated with the same `Emulator` that `scripts/play.py` uses. Panels: 2D world · keys pressed (the model's
action) · status · real observation waterfall · the model's predicted/imagined waterfall(s).

**Teacher-forced** first (one-step prediction — this is the flattering view), then **closed loop**, where the
model consumes only its own predictions after a 15-frame warm-up and the observer dreams alongside it on the same
action sequence. In the closed-loop pair, note that the actor's and observer's dreams are nearly identical —
smooth, well-separated, death-free — while *reality* fills with collisions and rebirth noise.

In [ ]:
# [5] Animations for both runs, teacher-forced then closed-loop.
for tag, lab in RUNS:
    for kind, note in [('teacherforced', 'TEACHER-FORCED (sees the real observation each step)'),
                       ('closedloop', 'CLOSED LOOP (own predictions; observer dreams alongside)')]:
        p = ANIM / f'{tag}_{kind}.gif'
        if p.exists():
            display(Markdown(f'**{lab} — {note}**  \n`{p.resolve()}`'))
            display(Image(filename=str(p)))

## Verdict (2026-07-29)

**The missing action pathway was a genuine bug, but it is NOT the dominant cause of the closed-loop collapse.**

- Cell [2] proves the old architecture's state was *literally* unaffected by its own action (‖Δh‖ = 0.0000).
- Fixing it leaves **teacher-forced behaviour identical** (2.8 deaths per 1000 frames both ways) — expected, since
  teacher forcing supplies the action's consequence for free via the real observation.
- Closed-loop improves only from **85.0 → 72.2** deaths per 1000 frames (~15%), still barely better than the
  **no-goal control at 79.1** — i.e. acting on its own imagination remains about as bad as having no policy.
- Fig 2: the imagination reaches **random-independent-frame error within ~10–20 steps**. It does not degrade
  gracefully, it **decouples** — which is why the actor's dream looks healthy (objects separated, no deaths) while
  reality has it dying repeatedly.

**Reading.** The remaining suspects are, in order: (1) **no latent-space consistency objective** — nothing ties the
imagined latent to observation-informed latents, which is exactly what RSSM's KL(posterior‖prior) provides;
(2) the **hidden-state reset every 48 frames**, still present in both runs (so this null is partially confounded);
(3) a **5-step imagination horizon trained versus 100+ evaluated**. The needed fix looks like a *training signal*
rather than more plumbing — which is the motivation for moving to RSSM, with the actor/observer contrast kept
*inside* RSSM so agency and architecture remain separable.